# Diamond frontier4-cache on WikiText-103

This notebook trains the **frontier4-cache** architecture (frontier4 sentence tower + causal first-word cache) on the SHA-pinned **WikiText-103 raw** corpus from NaNROUTER.

FineWeb is **not published in NaNROUTER** (the only FineWeb asset is a checkpoint release), so the fallback is WikiText-103. If you have a FineWeb text directory, point `CORPUS_DIR` at it and skip the Wiki download/split cells; the engine reads `.txt` and `.jsonl` documents.

The checkpoint and metrics are written to Google Drive after every epoch, so a Colab disconnect is resumable: rerun the setup cells and the training cell.


In [ ]:
#@title Check GPU and TensorFlow
!nvidia-smi || true
import tensorflow as tf
print("TensorFlow", tf.__version__)
print("GPUs", tf.config.list_physical_devices("GPU"))
# If the engine hits a TensorFlow/Keras API mismatch, install the exact
# GitHub-tested stack and restart the runtime:
# !pip -q install "tensorflow==2.16.2" "keras==3.0.5" "numpy==1.26.4"


In [ ]:
#@title Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
#@title Clone the engine branch
%cd /content
!rm -rf nanlm_experimental
!git clone --branch colab-wikitext --depth 1 https://github.com/shemshallah/nanlm_experimental.git
%cd /content/nanlm_experimental
!git log --oneline -1


In [ ]:
#@title Download WikiText-103 raw (540 MB, skipped if already present)
!if [ ! -s /content/wiki.train.raw ]; then \
  wget -q --show-progress -O /content/wiki.train.raw \
    https://github.com/shemshallah/NaNROUTER/releases/download/wikitext103/wiki.train.raw; \
fi
!ls -lh /content/wiki.train.raw


In [ ]:
#@title Download WikiText-103 raw (540 MB, skipped if already present)
!if [ ! -s /content/wiki.train.raw ]; then \
  wget -q --show-progress -O /content/wiki.train.raw \
    https://github.com/shemshallah/NaNROUTER/releases/download/wikitext103/wiki.train.raw; \
fi
!ls -lh /content/wiki.train.raw


## Optional smoke test
Run this once before the long job to catch a runtime/API problem. It uses a separate local state directory and a 5-minute budget.


In [ ]:
# Optional 5-minute smoke test (separate state, not the main checkpoint)
!STATE_DIR=/content/diamond_smoke \
 CORPUS_DIR=/content/wiki_corpus \
 EPOCHS=1 EXAMPLES=512 CORPUS_CHARS=500000 \
 TIME_BUDGET=5 TEST_SIZE=256 BATCH=32 \
 bash colab/train_wikitext.sh


In [ ]:
#@title Train frontier4-cache (resumable, 90-minute session)
# Every setting is an environment variable; edit the ones you need.
!STATE_DIR=/content/drive/MyDrive/diamond_wiki/state \
 CORPUS_DIR=/content/wiki_corpus \
 EPOCHS=100 \
 EXAMPLES=15000 \
 CORPUS_CHARS=50000000 \
 TIME_BUDGET=90 \
 TEST_SIZE=4096 \
 BATCH=32 \
 SENT_WEIGHT=0.8 \
 bash colab/train_wikitext.sh


In [ ]:
#@title Report the best and latest metrics
!python colab/report_metrics.py \
  /content/drive/MyDrive/diamond_wiki/state/diamond_metrics.json


## Resume after a disconnect

1. Reconnect the Colab runtime.
2. Run the Drive-mount, clone, download, and split cells again (they are idempotent; the split cell rewrites the same document files).
3. Run the training cell again. The engine loads `diamond_state.npz` from Drive and continues from the last completed epoch.

Keep `CORPUS_CHARS`, `TEST_SIZE`, model dimensions, and the corpus split unchanged between sessions; otherwise the checkpoint signature changes and training starts fresh.

`TIME_BUDGET=90` stops gracefully after an epoch and saves state. Reduce it if your Colab session is shorter. `EPOCHS=100` is the target total, not a per-session count.
